## Import ##

In [1]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pickle
from time import gmtime, strftime

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

/media/osero/SamsungSSD/miniconda_files/conda/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
# Load the state_dict into the model
import torch.nn as nn

class VideoClassifierLSTM(nn.Module):
    def __init__(self, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.fc = nn.Linear(self.dino_model.embed_dim, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):        
        dino_feature = self.dino_model(x)
        output = self.dropout(dino_feature)
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
video_lassifier_model = VideoClassifierLSTM(num_classes=744)

video_lassifier_model.load_state_dict(torch.load("/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/FINE_TUNED_LEFT_MODEL2024-12-30_08-17-42_1.pth"))

model = video_lassifier_model.dino_model
model.to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:45: UserWarning: xFormers is disabled (SwiGLU)
  warnings.warn("xFormers is disabled (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:29: UserWarning: xFormers is disabled (Attention)
  warnings.warn("xFormers is disabled (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:35: UserWarning: xFormers is disabled (Block)
  warnings.warn("xFormers is disable

## Functions ##

In [3]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Custom Dataset for a list of image paths
class ImageDataset(Dataset):
    def __init__(self):
        video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256" 
        full_sample_folder_list = []
        label_list = []
        for label_folder in sorted(os.listdir(video_folder)):
            full_label_folder = os.path.join(video_folder, label_folder)
            label = int(label_folder)
            for sample_folder in sorted(os.listdir(full_label_folder)):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                full_sample_folder_list.append(full_sample_folder)
                label_list.append(label)
        self.full_sample_folder_list = full_sample_folder_list
        self.label_list = label_list

    def __len__(self):
        return len(self.full_sample_folder_list)

    def __getitem__(self, idx):
        full_sample_folder = self.full_sample_folder_list[idx]
        image_list = []
        if (self.label_list[idx] < 613):
            return full_sample_folder, image_list, self.label_list[idx]
        
        for image_file in sorted(os.listdir(full_sample_folder)):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            transformed_image = transform(image)
            image_list.append(transformed_image)
        return full_sample_folder, image_list, self.label_list[idx]

In [4]:
dataset = ImageDataset()
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=8)

In [5]:
import gc

video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
video_paths = []
check_counter = 0
with torch.no_grad():  # Disable gradient calculation
    for paths, inputs, labels in dataloader:       
        if (labels[0] < 613):
            continue
        check_counter += 1
        inputs_stack = torch.stack(inputs).to(device).squeeze(1)
        outputs = model(inputs_stack)
        outputs_numpy = outputs.cpu().numpy().tolist()
        video_paths.append(paths)
        video_embeddings.append(outputs_numpy)
        video_labels.append(labels)
        if (check_counter % 1000) == 0:
            print(paths[0])
            print(strftime("%Y-%m-%d_%H-%M-%S", gmtime()))
            gc.collect()
            with open('features_face_frames_left_hand_small_finetuned_saved2.pickle', 'wb') as handle:
                pickle.dump((video_paths, video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('features_face_frames_left_hand_small_finetuned.pickle', 'wb') as handle:
    pickle.dump((video_paths, video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

aaa = 4



/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256/0649/User_4_001
2024-12-31_08-57-24
/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256/0687/User_4_006
2024-12-31_09-02-48
/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256/0722/User_7_004
2024-12-31_09-08-33


## Process ##

## Evaluation ##

In [ ]:
# with open('features_face_frames_small.pickle', 'wb') as handle:
#     pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

In [2]:
pickle_file11 = open('features_face_frames_left_hand_small_finetuned_saved.pickle', 'rb')
paths11, features11,labels11 = pickle.load(pickle_file11)
cc = 5

pickle_file22 = open('features_face_frames_left_hand_small_finetuned_saved2.pickle', 'rb')
paths22, features22,labels22 = pickle.load(pickle_file22)
cc = 5

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# Custom Dataset for a list of image paths
class ImageDataset(Dataset):
    def __init__(self):
        video_folder = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256" 
        full_sample_folder_list = []
        label_list = []
        for label_folder in sorted(os.listdir(video_folder)):
            full_label_folder = os.path.join(video_folder, label_folder)
            label = int(label_folder)
            for sample_folder in sorted(os.listdir(full_label_folder)):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                full_sample_folder_list.append(full_sample_folder)
                label_list.append(label)
        self.full_sample_folder_list = full_sample_folder_list
        self.label_list = label_list

    def __len__(self):
        return len(self.full_sample_folder_list)

    def __getitem__(self, idx):
        full_sample_folder = self.full_sample_folder_list[idx]
        image_list = []
        for image_file in sorted(os.listdir(full_sample_folder)):
            full_image_file = os.path.join(full_sample_folder, image_file)
            image = Image.open(full_image_file)
            transformed_image = transform(image)
            image_list.append(transformed_image)
        return full_sample_folder, image_list, self.label_list[idx]

In [ ]:
dataset = ImageDataset()
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=8)

In [ ]:
import gc

video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []
video_paths = []
check_counter = 0
with torch.no_grad():  # Disable gradient calculation
    for paths, inputs, labels in dataloader:
        check_counter += 1
        inputs_stack = torch.stack(inputs).to(device).squeeze(1)
        outputs = model(inputs_stack)
        outputs_numpy = outputs.cpu().numpy().tolist()
        video_paths.append(paths[0])
        video_embeddings.append(outputs_numpy)
        video_labels.append(int(labels[0]))
        if (check_counter % 1000) == 0:
            print(paths[0])
            print(strftime("%Y-%m-%d_%H-%M-%S", gmtime()))
            gc.collect()
            with open('features_face_frames_right_hand_small_finetuned_saved.pickle', 'wb') as handle:
                pickle.dump((video_paths, video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

with open('features_face_frames_right_hand_small_finetuned.pickle', 'wb') as handle:
    pickle.dump((video_paths, video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

aaa = 4

